In [ ]:
import os
import sys
import yaml
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import random
import shutil
from google.colab import drive, files

drive.mount('/content/drive')

SOURCE_DATASET = "/content/drive/MyDrive/"
LOCAL_DATASET = "/content/dataset"

Mounted at /content/drive


In [ ]:
os.makedirs(f"{LOCAL_DATASET}/images/train", exist_ok=True)
os.makedirs(f"{LOCAL_DATASET}/images/val", exist_ok=True)
os.makedirs(f"{LOCAL_DATASET}/labels/train", exist_ok=True)
os.makedirs(f"{LOCAL_DATASET}/labels/val", exist_ok=True)

for f in Path(f"{SOURCE_DATASET}/images/train").glob("*.jpg"):
    shutil.copy2(f, f"{LOCAL_DATASET}/images/train")
for f in Path(f"{SOURCE_DATASET}/images/val").glob("*.jpg"):
    shutil.copy2(f, f"{LOCAL_DATASET}/images/val")
for f in Path(f"{SOURCE_DATASET}/labels/train").glob("*.txt"):
    shutil.copy2(f, f"{LOCAL_DATASET}/labels/train")
for f in Path(f"{SOURCE_DATASET}/labels/val").glob("*.txt"):
    shutil.copy2(f, f"{LOCAL_DATASET}/labels/val")

print(f"Train images: {len(list(Path(LOCAL_DATASET+'/images/train').glob('*.jpg')))}")
print(f"Val images: {len(list(Path(LOCAL_DATASET+'/images/val').glob('*.jpg')))}")
print(f"Train labels: {len(list(Path(LOCAL_DATASET+'/labels/train').glob('*.txt')))}")
print(f"Val labels: {len(list(Path(LOCAL_DATASET+'/labels/val').glob('*.txt')))}")

Train images: 419
Val images: 51
Train labels: 419
Val labels: 51


In [ ]:
!pip install -q transformers timm

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

class YOLODataset(Dataset):
    def __init__(self, images_dir, labels_dir, transforms=None, img_size=800):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        self.img_size = img_size

        self.image_files = sorted([f.name for f in Path(images_dir).glob("*.jpg")])
        print(f"Найдено {len(self.image_files)} изображений в {images_dir}")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)

        # Загрузка изображения
        image = cv2.imread(img_path)
        if image is None:
            image = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = image.shape[:2]

        # Загрузка меток
        label_name = img_name.replace('.jpg', '.txt')
        label_path = os.path.join(self.labels_dir, label_name)

        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls = int(parts[0])
                        x_c, y_c, w, h = map(float, parts[1:5])
                        # Конвертируем в пиксели (x1, y1, x2, y2)
                        x1 = max(0.0, (x_c - w/2) * orig_w)
                        y1 = max(0.0, (y_c - h/2) * orig_h)
                        x2 = min(orig_w, (x_c + w/2) * orig_w)
                        y2 = min(orig_h, (y_c + h/2) * orig_h)
                        boxes.append([x1, y1, x2, y2])
                        labels.append(cls)

        boxes = np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4), dtype=np.float32)
        labels = np.array(labels, dtype=np.int64) if labels else np.zeros(0, dtype=np.int64)

        # Аугментации
        if self.transforms:
            transformed = self.transforms(image=image, bboxes=boxes.tolist(), class_labels=labels.tolist())
            image = transformed['image']
            boxes = np.array(transformed['bboxes'], dtype=np.float32) if transformed['bboxes'] else np.zeros((0, 4), dtype=np.float32)
            labels = np.array(transformed['class_labels'], dtype=np.int64) if transformed['class_labels'] else np.zeros(0, dtype=np.int64)

        # Конвертируем в формат DETR [cx, cy, w, h] нормализованный
        h, w = image.shape[1], image.shape[2]  # после ToTensor: C,H,W

        if len(boxes) > 0:
            # boxes в пикселях [x1, y1, x2, y2] -> нормализованные [cx, cy, w, h]
            boxes_detr = np.zeros_like(boxes)
            boxes_detr[:, 0] = ((boxes[:, 0] + boxes[:, 2]) / 2) / w  # cx
            boxes_detr[:, 1] = ((boxes[:, 1] + boxes[:, 3]) / 2) / h  # cy
            boxes_detr[:, 2] = (boxes[:, 2] - boxes[:, 0]) / w  # w
            boxes_detr[:, 3] = (boxes[:, 3] - boxes[:, 1]) / h  # h

            boxes_detr = np.clip(boxes_detr, 0.0, 1.0)
        else:
            boxes_detr = np.zeros((0, 4), dtype=np.float32)

        target = {
            'boxes': torch.from_numpy(boxes_detr).float(),
            'labels': torch.from_numpy(labels).long(),
            'image_id': torch.tensor([idx]),
            'orig_size': torch.tensor([orig_h, orig_w]),
            'size': torch.tensor([h, w])
        }

        return image, target, img_name

def collate_fn(batch):
    images, targets, names = zip(*batch)
    images = torch.stack(images, 0)
    return images, list(targets), list(names)

In [ ]:
train_transforms = A.Compose([
    A.Resize(height=800, width=800),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.1),
    A.Affine(scale=(0.9, 1.1), p=0.2),
    A.MotionBlur(blur_limit=7, p=0.2),
    A.Rotate(limit=10, p=0.2),
    A.Rotate(limit=10, p=0.3),
    A.Affine(scale=(0.9, 1.1), p=0.2),
    A.GaussianBlur(blur_limit=3, p=0.15),
    A.VerticalFlip(p=0.1),
    A.Perspective(scale=(0.05, 0.1), p=0.2),
    A.Transpose(p=0.2),
    A.RandomFog(p=0.15),
    A.RandomRain(p=0.1),
    A.ISONoise(p=0.1),
    A.MotionBlur(blur_limit=7, p=0.2),
    A.MedianBlur(blur_limit=5, p=0.1),
    A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.2),
    A.Emboss(alpha=(0.2, 0.5), strength=(0.2, 0.7), p=0.1),
    A.CoarseDropout(max_holes=8, max_height=20, max_width=20, min_holes=1, p=0.15),
    A.RandomShadow(p=0.15),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'], min_visibility=0.3))

val_transforms = A.Compose([
    A.Resize(height=800, width=800),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'], min_visibility=0.3))

set_seed(42)

train_dataset = YOLODataset(
    f"{LOCAL_DATASET}/images/train",
    f"{LOCAL_DATASET}/labels/train",
    transforms=train_transforms
)

val_dataset = YOLODataset(
    f"{LOCAL_DATASET}/images/val",
    f"{LOCAL_DATASET}/labels/val",
    transforms=val_transforms
)

# DataLoader'ы
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=0, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Найдено 419 изображений в /content/dataset/images/train
Найдено 51 изображений в /content/dataset/images/val
Train batches: 210, Val batches: 26


/tmp/ipykernel_2975/1187678537.py:22: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=20, max_width=20, min_holes=1, p=0.15),


In [ ]:
def compute_iou(box1, box2):
    """IoU между двумя боксами в формате [cx, cy, w, h] нормализованные"""
    x1_min = box1[0] - box1[2] / 2
    y1_min = box1[1] - box1[3] / 2
    x1_max = box1[0] + box1[2] / 2
    y1_max = box1[1] + box1[3] / 2

    x2_min = box2[0] - box2[2] / 2
    y2_min = box2[1] - box2[3] / 2
    x2_max = box2[0] + box2[2] / 2
    y2_max = box2[1] + box2[3] / 2

    inter_xmin = max(x1_min, x2_min)
    inter_ymin = max(y1_min, y2_min)
    inter_xmax = min(x1_max, x2_max)
    inter_ymax = min(y1_max, y2_max)

    if inter_xmax < inter_xmin or inter_ymax < inter_ymin:
        return 0.0

    inter_area = (inter_xmax - inter_xmin) * (inter_ymax - inter_ymin)
    box1_area = box1[2] * box1[3]
    box2_area = box2[2] * box2[3]
    union_area = box1_area + box2_area - inter_area

    return inter_area / union_area if union_area > 0 else 0.0

def compute_metrics_batch(predictions, targets, threshold=0.5, iou_threshold=0.5):
    """Считает FPR, Recall, Precision на батче"""
    fp = 0
    tn = 0
    tp = 0
    fn = 0

    for pred, target in zip(predictions, targets):
        has_true_objects = len(target['boxes']) > 0

        pred_scores = pred['scores'].cpu().numpy() if isinstance(pred['scores'], torch.Tensor) else pred['scores']
        pred_boxes = pred['boxes'].cpu().numpy() if isinstance(pred['boxes'], torch.Tensor) else pred['boxes']
        target_boxes = target['boxes'].cpu().numpy() if isinstance(target['boxes'], torch.Tensor) else target['boxes']

        pred_has_objects = np.sum(pred_scores > threshold) > 0

        # Считаем FPR (по изображениям, нет ли засорения)
        if not has_true_objects and not pred_has_objects:
            tn += 1
        elif not has_true_objects and pred_has_objects:
            fp += 1

        # Считаем TP, FN (по объектам)
        if len(target_boxes) > 0:
            matched = set()
            for pred_box in pred_boxes:
                best_iou = 0
                best_idx = -1
                for target_idx, target_box in enumerate(target_boxes):
                    if target_idx in matched:
                        continue
                    iou = compute_iou(pred_box, target_box)
                    if iou > best_iou:
                        best_iou = iou
                        best_idx = target_idx

                if best_iou > iou_threshold:
                    tp += 1
                    matched.add(best_idx)

            fn += len(target_boxes) - len(matched)

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    return {
        'fpr': fpr,
        'recall': recall,
        'precision': precision,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn
    }

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    """Удаляет overlapping боксы"""
    if len(boxes) == 0:
        return torch.tensor([], dtype=torch.long)

    # Конвертируем в [x1, y1, x2, y2] если нужно
    x1 = boxes[:, 0] - boxes[:, 2] / 2
    y1 = boxes[:, 1] - boxes[:, 3] / 2
    x2 = boxes[:, 0] + boxes[:, 2] / 2
    y2 = boxes[:, 1] + boxes[:, 3] / 2

    area = (x2 - x1) * (y2 - y1)
    _, order = scores.sort(descending=True)

    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)

        if len(order) == 1:
            break

        xx1 = torch.max(x1[i], x1[order[1:]])
        yy1 = torch.max(y1[i], y1[order[1:]])
        xx2 = torch.min(x2[i], x2[order[1:]])
        yy2 = torch.min(y2[i], y2[order[1:]])

        w = torch.clamp(xx2 - xx1, min=0)
        h = torch.clamp(yy2 - yy1, min=0)
        inter = w * h

        iou = inter / (area[i] + area[order[1:]] - inter)
        idx = (iou <= iou_threshold).nonzero(as_tuple=True)[0]
        order = order[idx + 1]

    return torch.tensor(keep, dtype=torch.long)

In [ ]:
from transformers import DetrForObjectDetection, DetrImageProcessor

class DETROreDetector(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.num_classes = num_classes
        self.model = DetrForObjectDetection.from_pretrained(
            "facebook/detr-resnet-101",
            num_labels=num_classes + 1,
            ignore_mismatched_sizes=True
        )
        self.processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-101")

    def forward(self, pixel_values, labels=None):
        return self.model(pixel_values=pixel_values, labels=labels)

    def predict(self, images, threshold=0.25, apply_nms=True, nms_threshold=0.4):
        self.eval()
        with torch.no_grad():
            outputs = self.model(pixel_values=images)

        probas = outputs.logits.softmax(-1)[..., :-1]
        batch_size = images.shape[0]
        results = []

        for i in range(batch_size):
            scores, labels = probas[i].max(-1)
            keep = scores > threshold

            boxes_filtered = outputs.pred_boxes[i][keep]
            scores_filtered = scores[keep]
            labels_filtered = labels[keep]

            if apply_nms and len(boxes_filtered) > 0:
                keep_nms = nms(boxes_filtered, scores_filtered, nms_threshold)
                boxes_filtered = boxes_filtered[keep_nms]
                scores_filtered = scores_filtered[keep_nms]
                labels_filtered = labels_filtered[keep_nms]

            results.append({
                'scores': scores_filtered,
                'labels': labels_filtered,
                'boxes': boxes_filtered
            })
        return results

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

model = DETROreDetector(num_classes=2).to(device)

Device: cuda
GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/243M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/179M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2586: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/batchnorm.py:133: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  super()._load_from_state_dict(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/batchnorm.py:133: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, whi

Loading weights:   0%|          | 0/785 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-101
Key                                                                         | Status     |                                                                                      
----------------------------------------------------------------------------+------------+--------------------------------------------------------------------------------------
model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                      
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                      
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                      
model.backbone.conv_encoder.model.layer1.0.downsa

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

In [ ]:
param_dicts = [
    {
        "params": [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad],
        "lr": 1e-4
    },
    {
        "params": [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad],
        "lr": 1e-5
    },
]

optimizer = AdamW(param_dicts, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=40, gamma=0.1)

num_epochs = 250
best_loss = float('inf')
best_recall = 0
patience = 50
patience_counter = 0

print(f"Начало обучения на {num_epochs} эпох...")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch} [Train]')
    for images, targets, _ in pbar:
        images = images.to(device)

        targets_detr = []
        for t in targets:
            targets_detr.append({
                'boxes': t['boxes'].to(device),
                'class_labels': t['labels'].to(device)
            })

        outputs = model(images, labels=targets_detr)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_train_loss = total_loss / len(train_loader)

    # VAL
    model.eval()
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        pbar = tqdm(val_loader, desc=f'Epoch {epoch} [Val]')
        for images, targets, _ in pbar:
            images = images.to(device)
            predictions = model.predict(images, threshold=0.5, apply_nms=True, nms_threshold=0.5)

            all_predictions.extend(predictions)
            all_targets.extend(targets)

    val_metrics = compute_metrics_batch(all_predictions, all_targets, threshold=0.5, iou_threshold=0.5)

    print(f"\nEpoch {epoch}:")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val FPR:    {val_metrics['fpr']:.4f}")
    print(f"  Val Recall: {val_metrics['recall']:.4f}")
    print(f"  Val Prec:   {val_metrics['precision']:.4f}")
    print(f"  TP: {val_metrics['tp']}, FP: {val_metrics['fp']}, FN: {val_metrics['fn']}, TN: {val_metrics['tn']}")

    if avg_train_loss < best_loss:
        best_loss = avg_train_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
            'metrics': val_metrics
        }, '/content/best_model.pth')
        print(f"  Best model saved (loss={best_loss:.4f}, FPR={val_metrics['fpr']:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1

    scheduler.step()

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        break


files.download('/content/best_model.pth')

Начало обучения на 250 эпох...


Epoch 0 [Val]: 100%|██████████| 26/26 [00:05<00:00,  4.53it/s]



Epoch 0:
  Train Loss: 1.3911
  Val FPR:    0.5000
  Val Recall: 0.2000
  Val Prec:   0.8333
  TP: 10, FP: 2, FN: 40, TN: 2
  Best model saved (loss=1.3911, FPR=0.5000)


Epoch 1 [Val]: 100%|██████████| 26/26 [00:05<00:00,  4.72it/s]



Epoch 1:
  Train Loss: 1.2581
  Val FPR:    0.5000
  Val Recall: 0.4000
  Val Prec:   0.9091
  TP: 20, FP: 2, FN: 30, TN: 2
  Best model saved (loss=1.2581, FPR=0.5000)


...

Epoch 107 [Val]: 100%|██████████| 26/26 [00:05<00:00,  4.66it/s]



Epoch 107:
  Train Loss: 0.5938
  Val FPR:    0.7500
  Val Recall: 0.7200
  Val Prec:   0.9231
  TP: 36, FP: 3, FN: 14, TN: 1


Epoch 108 [Val]: 100%|██████████| 26/26 [00:05<00:00,  4.70it/s]



Epoch 108:
  Train Loss: 0.5722
  Val FPR:    0.7500
  Val Recall: 0.7200
  Val Prec:   0.9231
  TP: 36, FP: 3, FN: 14, TN: 1


Epoch 109 [Train]:  78%|███████▊  | 163/210 [02:14<00:40,  1.16it/s, loss=0.3764]